# Regression Splines – interaktives Colab-Notebook

**Ziel:** Eine kubische Regressionsspline für eine numerische Eingabevariable trainieren, Knotenpunkte verändern, die Kurve visualisieren und mit linearer und polynomialer Regression vergleichen.

Beispiel: **Advertising-Datensatz** aus *An Introduction to Statistical Learning*: `TV` → `sales`. TV-Werbeausgaben sind in **1.000 US-Dollar**, `sales` in **1.000 verkauften Einheiten** angegeben.

**Fachliche Idee:** Spline-Funktionen verwenden mehrere polynomiale Abschnitte, deren Übergänge glatt sind. Die *Knotenpunkte* (`knots`) bestimmen, wo die Abschnitte aufeinandertreffen. **Die Knoten sind eine Modellentscheidung; die Regressionskoeffizienten werden aus den Trainingsdaten gelernt.**

## 1 · Bibliotheken importieren
Verwendet `SplineTransformer` aus scikit-learn (kubische B-Spline-Basis) und `LinearRegression`. Kein zusätzliches Paket erforderlich in üblichen Colab-Umgebungen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import SplineTransformer, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Bibliotheken geladen.")

## 2 · CSV-Datensatz laden
In Colab erscheint ein Upload-Dialog. Wähle z. B. `Advertising(1).csv`. Beim lokalen Ausführen wird nach einer vorhandenen CSV-Datei gesucht.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Bitte eine CSV-Datei hochladen.")
    filename = next(iter(uploaded))
except ImportError:
    candidates = [Path('Advertising(1).csv'), Path('Advertising.csv')]
    filename = str(next((p for p in candidates if p.exists()), 'Advertising(1).csv'))

df = pd.read_csv(filename)
# Optional: unbenannte Indexspalte aus CSV-Export entfernen
index_cols = [col for col in df.columns if str(col).startswith('Unnamed:')]
df = df.drop(columns=index_cols)
print(f"Datensatz: {filename} | {len(df)} Zeilen")
print("Spalten:", list(df.columns))
display(df.head())

## 3 · Einstellungen: Feature, Zielvariable, Splines
**Nur diese Zelle ändern**, um andere Datensätze oder Hypothesenräume zu testen.

- `FEATURE`: numerische Eingabevariable (z. B. `TV`, `Age`)
- `TARGET`: numerische Zielvariable (z. B. `sales`, `MarketValue`)
- `DEGREE = 3`: kubische Splines
- `KNOT_MODE = 'manual'`: Knoten selbst vorgeben (in der Einheit des Features)
- `KNOT_MODE = 'quantile'`: Knoten werden **nur aus Trainingsdaten** automatisch bestimmt
- `MANUAL_KNOTS`: **innere** Knotenpunkte; keine Endpunkte nötig


In [ ]:
FEATURE = 'TV'
TARGET = 'sales'
DEGREE = 3
KNOT_MODE = 'manual'    # 'manual' oder 'quantile'
MANUAL_KNOTS = [75, 150, 225]  # TV wird in Tausend US-Dollar gemessen
N_KNOTS_AUTO = 5        # Anzahl der Knoten inkl. Randknoten bei quantile
TEST_SIZE = 0.20
RANDOM_STATE = 42

for col in (FEATURE, TARGET):
    if col not in df.columns:
        raise ValueError(f"Spalte '{col}' fehlt. Vorhanden: {list(df.columns)}")

data = df[[FEATURE, TARGET]].copy()
for col in (FEATURE, TARGET):
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.dropna().reset_index(drop=True)
if len(data) < 15:
    raise ValueError('Für diese Übung werden mindestens 15 vollständige Beobachtungen benötigt.')

X = data[[FEATURE]]
y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f'Training: {len(X_train)} | Test: {len(X_test)}')
print(f'Beobachteter Wertebereich von {FEATURE}: {X[FEATURE].min():.2f} bis {X[FEATURE].max():.2f}')

## 4 · Spline-Modell definieren und trainieren

`SplineTransformer` erzeugt Basisfunktionen. Anschließend schätzt eine gewöhnliche lineare Regression die Koeffizienten für diese Basisfunktionen. **Trotz gekrümmter Vorhersagekurve bleibt die Regression linear in ihren Koeffizienten.**

`extrapolation='constant'` bedeutet: Außerhalb des beobachteten Bereichs ist die Vorhersage konstant fortgesetzt; für Extrapolation ist besondere Vorsicht nötig.

In [ ]:
if KNOT_MODE == 'manual':
    knots = np.sort(np.asarray(MANUAL_KNOTS, dtype=float))
    if len(knots) == 0 or np.any(np.diff(knots) <= 0):
        raise ValueError('MANUAL_KNOTS benötigt mindestens einen Wert; alle müssen verschieden sein.')
    if knots[0] <= X_train[FEATURE].min() or knots[-1] >= X_train[FEATURE].max():
        raise ValueError('Die manuellen Knoten müssen innerhalb des Trainings-Wertebereichs liegen.')
    # SplineTransformer erwartet hier alle Knoten: Randknoten + innere Knoten
    all_knots = np.r_[X_train[FEATURE].min(), knots, X_train[FEATURE].max()]
    spline_basis = SplineTransformer(
        degree=DEGREE, knots=all_knots.reshape(-1, 1),
        include_bias=False, extrapolation='constant'
    )
elif KNOT_MODE == 'quantile':
    spline_basis = SplineTransformer(
        degree=DEGREE, knots='quantile', n_knots=N_KNOTS_AUTO,
        include_bias=False, extrapolation='constant'
    )
else:
    raise ValueError("KNOT_MODE muss 'manual' oder 'quantile' sein")

spline_model = make_pipeline(spline_basis, LinearRegression())
spline_model.fit(X_train, y_train)
actual_knots = np.unique(spline_model.named_steps['splinetransformer'].bsplines_[0].t[DEGREE:-DEGREE])
print('Verwendete Knoten (inkl. Randknoten):', np.round(actual_knots, 2))
print('Anzahl Spline-Basisfunktionen:', spline_model.named_steps['splinetransformer'].n_features_out_)

## 5 · Scatterplot und trainierte Spline-Kurve
Orange Punkte sind die Beobachtungen. Die blaue Kurve ist das **trainierte** Spline-Modell; gestrichelte Linien markieren **innere Knoten**.

In [ ]:
x_grid = np.linspace(X[FEATURE].min(), X[FEATURE].max(), 500)
x_grid_df = pd.DataFrame({FEATURE: x_grid})
y_grid = spline_model.predict(x_grid_df)

plt.figure(figsize=(10, 5.5))
plt.scatter(X_train[FEATURE], y_train, alpha=.65, s=23, label='Trainingsdaten')
plt.scatter(X_test[FEATURE], y_test, alpha=.65, s=23, marker='x', label='Testdaten')
plt.plot(x_grid, y_grid, linewidth=2.5, label=f'Spline Grad {DEGREE}')
for i, knot in enumerate(actual_knots[1:-1]):
    plt.axvline(knot, color='gray', linestyle='--', alpha=.55,
                label='Innerer Knoten' if i == 0 else None)
plt.xlabel(FEATURE)
plt.ylabel(TARGET)
plt.title('Regressionsspline: Anpassung und Knotenpunkte')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 6 · Mit linearer und polynomialer Regression vergleichen
Alle drei Modelle verwenden **dieselbe Eingabevariable**, dieselben Trainings- und Testdaten und unterschiedliche Funktionsklassen.

- **Linear:** Gerade
- **Polynomial, Grad 3:** Ein einziges kubisches Polynom über den gesamten Wertebereich
- **Spline, Grad 3:** Mehrere glatt verbundene kubische Polynomabschnitte

In [ ]:
linear_model = LinearRegression().fit(X_train, y_train)
poly_model = make_pipeline(PolynomialFeatures(degree=3, include_bias=False),
                           LinearRegression()).fit(X_train, y_train)
models = {
    'Linear': linear_model,
    'Polynomial (Grad 3)': poly_model,
    f'Spline (Grad {DEGREE})': spline_model
}

plt.figure(figsize=(10, 5.5))
plt.scatter(X_train[FEATURE], y_train, alpha=.4, s=22, label='Trainingsdaten')
for name, model in models.items():
    plt.plot(x_grid, model.predict(x_grid_df), linewidth=2.2, label=name)
for knot in actual_knots[1:-1]:
    plt.axvline(knot, color='gray', linestyle=':', alpha=.35)
plt.xlabel(FEATURE)
plt.ylabel(TARGET)
plt.title('Hypothesenräume im Vergleich')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 7 · Modellgüte: R², MAE, MSE, RMSE und SSE (RSS)
Die Kennzahlen werden **auf den Testdaten** berechnet. Bei identischem Testset sind sie direkt vergleichbar. MSE und SSE haben quadrierte Zielvariablen-Einheiten.

In [ ]:
results = []
for name, model in models.items():
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    results.append({
        'Modell': name,
        'R²': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'SSE / RSS': np.sum((y_test.to_numpy() - pred) ** 2)
    })
comparison = pd.DataFrame(results).set_index('Modell')
display(comparison.round(3))

## 8 · Optional: Cross-Validation für Anzahl der Knoten
Die Zahl der Knoten beeinflusst die Flexibilität. Wir testen mehrere Möglichkeiten mit **5-facher Cross-Validation auf den Trainingsdaten**. Der Testdatensatz bleibt für die abschließende Bewertung unangetastet.

Für den Vergleich verwendet der `SplineTransformer` automatisch **Quantilknoten**, die in jeder Trainingsfalte ermittelt werden; so werden Informationen aus Validierungsfalten nicht vorweggenommen.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for n_knots in [3, 4, 5, 6, 8, 10]:
    candidate = make_pipeline(
        SplineTransformer(n_knots=n_knots, degree=3,
                          knots='quantile', include_bias=False,
                          extrapolation='constant'),
        LinearRegression()
    )
    neg_mse_scores = cross_val_score(candidate, X_train, y_train,
                                    scoring='neg_mean_squared_error', cv=cv)
    cv_rows.append({'Knoten (inkl. Randknoten)': n_knots,
                    'CV-RMSE': np.sqrt(-neg_mse_scores.mean())})
cv_table = pd.DataFrame(cv_rows)
display(cv_table.round(3))
plt.figure(figsize=(7, 4))
plt.plot(cv_table['Knoten (inkl. Randknoten)'], cv_table['CV-RMSE'], marker='o')
plt.xlabel('Anzahl Knoten (inkl. Randknoten)')
plt.ylabel('CV-RMSE')
plt.title('Spline-Komplexität und Validierungsfehler')
plt.grid(alpha=.25)
plt.show()

## 9 · Übung / Reflexion

1. Ändere `MANUAL_KNOTS` von `[75, 150, 225]` auf z. B. `[100, 200]`. Wie verändert sich die Spline-Kurve?
2. Ändere `KNOT_MODE` zu `'quantile'` und teste `N_KNOTS_AUTO = 4`, `6`, `10`.
3. Vergleiche den kubischen Spline mit dem **globalen** kubischen Polynom. Wo unterscheiden sich die Kurven?
4. Vergleiche Test-RMSE und CV-RMSE. Warum ist eine glattere oder flexiblere Kurve nicht automatisch besser?
5. Verwende einen anderen Datensatz. Passe `FEATURE` und `TARGET` an. Falls du manuelle Knoten wählst, passe ihre Werte an die Skala des Features an.

**Merksatz:** Eine Spline ist eine stückweise polynomiale Funktion mit festgelegten Glattheitsbedingungen. Knots und Polynomgrad definieren die Modellklasse; die Koeffizienten werden aus Trainingsdaten geschätzt.

**Referenzen:** James et al., *An Introduction to Statistical Learning* (Kapitel *Moving Beyond Linearity*); [scikit-learn: SplineTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html).